![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
***
**This capstone integrates NB-01 through NB-07 into a single production-ready ML pipeline.**

**Pipeline stages:**
1. Data assembly, EDA, and train/val/test temporal splits
2. Preprocessing pipeline with leakage-safe transforms
3. Baseline and candidate model comparison (LR, RF, GBM, XGBoost)
4. Hyperparameter optimisation with Optuna (best model)
5. Comprehensive evaluation: AUC, AP, calibration, DCA, EPV
6. SHAP global and local explanations
7. Risk stratification and patient dashboard
8. MLOps-ready: model serialisation + drift detection template
9. Publication-quality summary figure + methods narrative

**Estimated time:** 4 hours | **Level:** Advanced


## 0. Setup & Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns
import os, warnings, json
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod04", exist_ok=True)

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, TimeSeriesSplit)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              roc_curve, precision_recall_curve, confusion_matrix,
                              classification_report)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
SEED = 42

# ── Build dataset ─────────────────────────────────────────────────────────────
np.random.seed(SEED); N=3500
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5+0.02*(age-62)/15),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
start=pd.Timestamp("2019-01-01"); np.random.seed(5)
admit_dates=[start+pd.Timedelta(days=int(d))
             for d in np.random.randint(0,(pd.Timestamp("2023-12-31")-start).days,N)]
df=pd.DataFrame({
    "patient_id":[f"PT-{i:05d}" for i in range(1,N+1)],
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,"ckd":ckd,
    "obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,
    "readmitted":readmitted,"admit_date":pd.to_datetime(admit_dates),
})
df=df.sort_values("admit_date").reset_index(drop=True)
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"
print(f"Dataset: {df.shape} | Positive rate: {df[TARGET].mean()*100:.1f}%")
print(f"Dates: {df.admit_date.min().date()} → {df.admit_date.max().date()}")


## 1. Temporal Train / Val / Test Split

In [ ]:
df_train = df[df.admit_date.dt.year <= 2021]
df_val   = df[df.admit_date.dt.year == 2022]
df_test  = df[df.admit_date.dt.year == 2023]
print("Temporal split:")
for name,sub in [("Train(≤2021)",df_train),("Val(2022)",df_val),("Test(2023)",df_test)]:
    print(f"  {name:12s}: N={len(sub):5d} | positive={sub[TARGET].mean()*100:.1f}%")

X_train=df_train[FEATURES]; y_train=df_train[TARGET]
X_val  =df_val[FEATURES];   y_val  =df_val[TARGET]
X_test =df_test[FEATURES];  y_test =df_test[TARGET]


## 2. Preprocessing Pipeline (Leakage-Safe)

In [ ]:
pre = ColumnTransformer([
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), NUMERIC),
    ("bin", SimpleImputer(strategy="most_frequent"), BINARY),
    ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                      ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), CATEGORIC),
])
pre.fit(X_train)  # FIT ONLY ON TRAIN
Xtr=pre.transform(X_train); Xval=pre.transform(X_val); Xte=pre.transform(X_test)
ohe_nm=pre.named_transformers_["cat"]["ohe"].get_feature_names_out(CATEGORIC)
feat_nm=NUMERIC+BINARY+list(ohe_nm)
Xtr_df=pd.DataFrame(Xtr,columns=feat_nm)
Xval_df=pd.DataFrame(Xval,columns=feat_nm)
Xte_df=pd.DataFrame(Xte,columns=feat_nm)
print(f"Features: {len(feat_nm)} | Train:{Xtr.shape} Val:{Xval.shape} Test:{Xte.shape}")


## 3. Candidate Model Race

In [ ]:
models_race = {
    "Logistic Reg (L2)"  : LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=SEED),
    "Random Forest"      : RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced",random_state=SEED,n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,subsample=0.8,random_state=SEED),
}
race_results=[]
for name,model in models_race.items():
    model.fit(Xtr,y_train)
    p_val =model.predict_proba(Xval)[:,1]
    p_test=model.predict_proba(Xte)[:,1]
    race_results.append({
        "Model":name,
        "Val AUC" :round(roc_auc_score(y_val, p_val),4),
        "Test AUC":round(roc_auc_score(y_test,p_test),4),
        "Val AP"  :round(average_precision_score(y_val,p_val),4),
        "Val Brier":round(brier_score_loss(y_val,p_val),4),
    })
race_df=pd.DataFrame(race_results).sort_values("Val AUC",ascending=False)
print("Model race results:"); display(race_df)
best_name=race_df.iloc[0]["Model"]
best_model=models_race[best_name]
print(f" Best model: {best_name}")


## 4. Optuna Fine-Tuning of Best Model

In [ ]:
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    skf=StratifiedKFold(3,shuffle=True,random_state=SEED)

    def objective(trial):
        p={"n_estimators":trial.suggest_int("n_estimators",100,400),
           "max_depth":trial.suggest_int("max_depth",3,8),
           "learning_rate":trial.suggest_float("learning_rate",0.01,0.3,log=True),
           "subsample":trial.suggest_float("subsample",0.5,1.0),
           "min_samples_leaf":trial.suggest_int("min_samples_leaf",5,30),
           "random_state":SEED}
        m=GradientBoostingClassifier(**p)
        return cross_val_score(m,Xtr,y_train,cv=skf,scoring="roc_auc").mean()

    study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective,n_trials=40,show_progress_bar=False)

    final_model=GradientBoostingClassifier(**study.best_params)
    final_model.fit(Xtr,y_train)
    p_opt_val =final_model.predict_proba(Xval)[:,1]
    p_opt_test=final_model.predict_proba(Xte)[:,1]
    print(f"Optuna best CV AUC : {study.best_value:.4f}")
    print(f"Optuna Val  AUC    : {roc_auc_score(y_val,p_opt_val):.4f}")
    print(f"Optuna Test AUC    : {roc_auc_score(y_test,p_opt_test):.4f}")
except ImportError:
    print("Optuna not available — using default GBM")
    final_model=best_model
    p_opt_test=final_model.predict_proba(Xte)[:,1]
    p_opt_val=final_model.predict_proba(Xval)[:,1]


## 5. Calibration with Platt Scaling

In [ ]:
from sklearn.linear_model import LogisticRegression as _LR
platt=_LR(C=1e6,max_iter=1000)
platt.fit(p_opt_val.reshape(-1,1),y_val.values)
p_cal=platt.predict_proba(p_opt_test.reshape(-1,1))[:,1]

brier_raw=brier_score_loss(y_test,p_opt_test)
brier_cal=brier_score_loss(y_test,p_cal)
auc_final=roc_auc_score(y_test,p_cal)
print(f"Final model (calibrated): AUC={auc_final:.4f} | Brier: {brier_raw:.4f}→{brier_cal:.4f}")


## 6. SHAP Global Explanations

In [ ]:
try:
    import shap
    explainer=shap.TreeExplainer(final_model)
    shap_vals=explainer.shap_values(Xte_df)
    mean_abs_shap=np.abs(shap_vals).mean(axis=0)
    top15=np.argsort(mean_abs_shap)[-15:][::-1]
    print("Top 5 features by mean |SHAP|:")
    for i in top15[:5]:
        print(f"  {feat_nm[i]:20s}: {mean_abs_shap[i]:.4f}")
    shap_ok=True
except Exception as e:
    print(f"SHAP: {e}"); shap_ok=False; shap_vals=None


## 7. Risk Stratification & Patient Dashboard

In [ ]:
def risk_tier(prob):
    if prob < 0.10:  return "Low (<10%)"
    elif prob < 0.20: return "Moderate (10-20%)"
    elif prob < 0.35: return "High (20-35%)"
    else:             return "Very high (>35%)"

df_results = df_test[["patient_id","age","sex","payer","hf","diabetes","ckd","los_days"]].copy()
df_results["pred_prob"] = np.round(p_cal,4)
df_results["risk_tier"] = df_results.pred_prob.apply(risk_tier)
df_results["y_true"]    = y_test.values

tier_stats = df_results.groupby("risk_tier",observed=True).agg(
    N=("patient_id","count"),
    actual_rate=("y_true","mean"),
    mean_prob=("pred_prob","mean"),
).round(3)
tier_stats[["actual_rate","mean_prob"]]*=100
print("Risk tier summary:")
display(tier_stats)
print(" Sample high-risk patients:")
display(df_results[df_results.risk_tier=="Very high (>35%)"].head(5).reset_index(drop=True))


## 8. Publication Summary Figure

In [ ]:
fig=plt.figure(figsize=(20,13))
fig.suptitle("End-to-End Clinical ML Pipeline — 30-Day Readmission (N=3,500)",fontsize=15,y=1.01)
gs=gridspec.GridSpec(3,4,figure=fig,hspace=0.45,wspace=0.35)

# A — Data split timeline
ax_a=fig.add_subplot(gs[0,:2])
monthly=(df.set_index("admit_date").resample("ME")[TARGET].agg(["mean","count"]))
colors_t=["#4878CF" if y<=2021 else "#6ACC65" if y==2022 else "#D65F5F"
          for y in monthly.index.year]
ax_a.bar(monthly.index,monthly["mean"]*100,color=colors_t,width=20,alpha=0.85)
from matplotlib.patches import Patch
ax_a.legend(handles=[Patch(color="#4878CF",label="Train≤2021"),
                      Patch(color="#6ACC65",label="Val 2022"),
                      Patch(color="#D65F5F",label="Test 2023")],fontsize=8)
ax_a.set_ylabel("Readmission rate (%)"); ax_a.set_title("A  Temporal split",fontweight="bold")
import matplotlib.dates as mdates
ax_a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax_a.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax_a.xaxis.get_majorticklabels(),rotation=25,ha="right")

# B — Model race
ax_b=fig.add_subplot(gs[0,2:])
x=np.arange(len(race_df)); w=0.35
ax_b.bar(x-w/2,race_df["Val AUC"],w,label="Val AUC",color="#4878CF",alpha=0.85)
ax_b.bar(x+w/2,race_df["Test AUC"],w,label="Test AUC",color="#D65F5F",alpha=0.85)
ax_b.set_xticks(x); ax_b.set_xticklabels(race_df["Model"],rotation=12,fontsize=9)
ax_b.set_ylabel("AUC-ROC"); ax_b.set_title("B  Model race",fontweight="bold")
ax_b.legend(fontsize=9); ax_b.set_ylim(0.5,0.85)
for i,(v,t) in enumerate(zip(race_df["Val AUC"],race_df["Test AUC"])):
    ax_b.text(i-w/2,v+0.003,f"{v:.3f}",ha="center",fontsize=8)
    ax_b.text(i+w/2,t+0.003,f"{t:.3f}",ha="center",fontsize=8)

# C — ROC
ax_c=fig.add_subplot(gs[1,0])
fpr_f,tpr_f,_=roc_curve(y_test,p_cal)
ax_c.plot(fpr_f,tpr_f,color="#1F4E79",lw=2,label=f"AUC={auc_final:.3f}")
ax_c.plot([0,1],[0,1],"k--",lw=1); ax_c.fill_between(fpr_f,tpr_f,alpha=0.1,color="#1F4E79")
ax_c.set_xlabel("1-Specificity"); ax_c.set_ylabel("Sensitivity")
ax_c.set_title("C  ROC",fontweight="bold"); ax_c.legend(fontsize=9)

# D — Calibration
ax_d=fig.add_subplot(gs[1,1])
fp_c,mp_c=calibration_curve(y_test,p_cal,n_bins=10)
ax_d.plot(mp_c,fp_c,"o-",color="#D65F5F",lw=2,ms=5,label=f"Calibrated (Brier={brier_cal:.3f})")
ax_d.plot([0,1],[0,1],"k--",lw=1)
ax_d.set_xlabel("Predicted"); ax_d.set_ylabel("Observed fraction")
ax_d.set_title("D  Calibration",fontweight="bold"); ax_d.legend(fontsize=8)

# E — Risk tier
ax_e=fig.add_subplot(gs[1,2])
tier_order=["Low (<10%)","Moderate (10-20%)","High (20-35%)","Very high (>35%)"]
tier_colors=["#4878CF","#6ACC65","#D4A017","#D65F5F"]
t_rates=[tier_stats.loc[t,"actual_rate"] if t in tier_stats.index else 0 for t in tier_order]
t_ns=[tier_stats.loc[t,"N"] if t in tier_stats.index else 0 for t in tier_order]
bars_e=ax_e.bar(range(4),t_rates,color=tier_colors,alpha=0.85)
for i,(v,n) in enumerate(zip(t_rates,t_ns)):
    ax_e.text(i,v+0.5,f"{v:.0f}% (n={n})",ha="center",fontsize=8)
ax_e.set_xticks(range(4)); ax_e.set_xticklabels(["Low","Mod","High","V.High"],fontsize=8)
ax_e.set_ylabel("Actual readmission rate (%)"); ax_e.set_title("E  Risk stratification",fontweight="bold")

# F — DCA
ax_f=fig.add_subplot(gs[1,3])
thrs_dca=np.linspace(0.01,0.50,80)
def net_benefit(yt,yp,t): 
    yp_b=(yp>=t).astype(int)
    tp=((yp_b==1)&(yt==1)).sum(); fp=((yp_b==1)&(yt==0)).sum()
    return (tp-fp*(t/(1-t)))/len(yt)
nb_model=[net_benefit(y_test.values,p_cal,t) for t in thrs_dca]
nb_all  =[net_benefit(y_test.values,np.ones(len(y_test)),t) for t in thrs_dca]
ax_f.plot(thrs_dca,nb_model,color="#1F4E79",lw=2,label="Model")
ax_f.plot(thrs_dca,nb_all,"k-.",lw=1.5,label="Treat all")
ax_f.axhline(0,color="gray",ls="--",lw=1,label="Treat none")
ax_f.set_xlabel("Threshold"); ax_f.set_ylabel("Net benefit")
ax_f.set_title("F  Decision Curve Analysis",fontweight="bold"); ax_f.legend(fontsize=8)
ax_f.set_xlim(0,0.5)

# G — SHAP bar
ax_g=fig.add_subplot(gs[2,:2])
if shap_ok:
    top_idx2=np.argsort(mean_abs_shap)[-12:]
    ax_g.barh([feat_nm[i] for i in top_idx2],mean_abs_shap[top_idx2],color="#4878CF",alpha=0.85)
    ax_g.set_xlabel("Mean |SHAP value|"); ax_g.set_title("G  SHAP feature importance",fontweight="bold")
else:
    fi=pd.Series(final_model.feature_importances_,index=feat_nm).sort_values().tail(12)
    ax_g.barh(fi.index,fi.values,color="#4878CF",alpha=0.85)
    ax_g.set_xlabel("Gini importance"); ax_g.set_title("G  Feature importance (MDI)",fontweight="bold")

# H — Score distribution
ax_h=fig.add_subplot(gs[2,2:])
ax_h.hist(p_cal[y_test==0],bins=30,alpha=0.6,color="#4878CF",label="Not readmitted",density=True)
ax_h.hist(p_cal[y_test==1],bins=30,alpha=0.6,color="#D65F5F",label="Readmitted",density=True)
ax_h.axvline(0.20,color="black",ls="--",lw=1.5,label="Threshold (20%)")
ax_h.set_xlabel("Predicted readmission probability"); ax_h.set_ylabel("Density")
ax_h.set_title("H  Score distribution by outcome",fontweight="bold"); ax_h.legend(fontsize=9)

plt.savefig("/tmp/mod04/capstone_summary.png",bbox_inches="tight",dpi=300); plt.show()
print("Saved at 300 dpi: capstone_summary.png")


## 9. MLOps — Model Serialisation & Drift Detection Template

In [ ]:
import pickle, hashlib

# Serialise the pipeline (preprocessor + calibrated model)
pipeline_obj = {"preprocessor":pre,"model":final_model,"calibrator":platt,
                "feature_names":feat_nm,"target":TARGET,
                "train_auc":roc_auc_score(y_val,p_opt_val),
                "test_auc":auc_final,"train_end":"2021-12-31","eval_date":"2023-12-31"}
with open("/tmp/mod04/readmission_pipeline.pkl","wb") as f:
    pickle.dump(pipeline_obj,f)
print("Model serialised to /tmp/mod04/readmission_pipeline.pkl")

# Drift detection: compare feature means between train and test
drift_report=[]
for col in NUMERIC:
    mean_tr=df_train[col].mean(); mean_te=df_test[col].mean()
    pct_drift=abs(mean_te-mean_tr)/max(abs(mean_tr),1e-8)*100
    drift_report.append({"Feature":col,"Train mean":round(mean_tr,2),
                          "Test mean":round(mean_te,2),"Drift %":round(pct_drift,1),
                          "Flag":">10%❗" if pct_drift>10 else "OK"})
drift_df=pd.DataFrame(drift_report)
print(" Feature drift report (train→test):")
display(drift_df)
print()
print("Recommendation: re-train if ≥3 features show >10% drift or AUC drops >0.03 on new data.")


## 10. Methods Narrative (Template)

> **Machine Learning Methods**
>
> We split the data temporally: training (2019–2021, N=XXX), validation (2022, N=XXX), and test (2023, N=XXX) to simulate prospective deployment. All preprocessing (median imputation, standardisation, one-hot encoding) was fit exclusively on training data to prevent leakage.
>
> We evaluated four candidate models: logistic regression (L2), random forest, gradient boosting (GBM), and XGBoost. The GBM was selected based on validation AUC and further tuned with 40-trial Bayesian optimisation (Optuna). Predicted probabilities were recalibrated on the validation set using Platt scaling; calibration was assessed visually and via Brier score.
>
> Discrimination was quantified by AUC-ROC and Average Precision (AUC-PR), both with 1000-sample bootstrap 95% CIs. Clinical utility was assessed by Decision Curve Analysis at thresholds 1–50%. Model explanations were generated with SHAP TreeExplainer; global importance was summarised by mean |SHAP value| and local explanations were produced for high-risk patients.
>
> The final model was serialised and a feature drift template implemented to monitor input distribution shifts in production. EPV = XX (events per variable), confirming adequate sample size for the XX-feature model.
>
> All analyses used Python 3.10 (sklearn 1.3, lightgbm 4.x, shap 0.44, optuna 3.x).


## 11. Final Knowledge Check
1. Your temporal test AUC (0.77) is 0.04 lower than random split AUC (0.81). What does this tell you about model deployment readiness?
2. The drift report flags `creatinine` with 14% drift between train and test. What should you do before redeploying?
3. EPV = 8.5 for your logistic regression with 12 predictors and 105 events. Is this adequate?
4. SHAP shows `hf` as the top predictor, but clinical team says `creatinine` should matter more. How would you investigate this?
5. Describe two failure modes of the Platt scaling recalibration approach used here.


In [ ]:
# Exercise 3 — EPV
n_events=y_test.sum()
n_params=12  # number of predictors
EPV=n_events/n_params
print(f"Events in test set : {n_events}")
print(f"Model parameters   : {n_params}")
print(f"EPV                : {EPV:.1f}")
print()
if EPV>=10:
    print("EPV ≥ 10: adequate (conventional rule of thumb)")
elif EPV>=5:
    print("EPV 5–10: borderline — consider penalised regression")
else:
    print("EPV < 5: risk of overfitting — use penalised LR or reduce predictors")


***
## Module 04 Complete ✓

**NB-01** Temporal splits, preprocessing pipelines, class imbalance, baselines, CV strategies  
**NB-02** Decision trees (bias-variance), random forest, MDI vs permutation importance  
**NB-03** XGBoost + LightGBM, early stopping, Optuna Bayesian tuning, learning curves  
**NB-04** AUC-ROC/PR with bootstrap CIs, calibration plots, Platt/isotonic recalibration, DCA  
**NB-05** Grid/random/halving search, nested CV, regularisation paths, overfitting diagnosis  
**NB-06** SHAP TreeExplainer, global beeswarm, local waterfall, dependence plots, clinical report  
**NB-07** Three production models: sepsis alert (high-recall), ICU LOS regression, readmission scoring  
**NB-08** Capstone: full pipeline → model race → Optuna → calibration → SHAP → DCA → MLOps

**Next → Module 05: NLP for Clinical Text**
